In [0]:
import json
from pyspark.sql.types import StructType

In [0]:
%run ./Initial

In [0]:
databases = ["bronze", "silver", "gold"]
base_query = "CREATE DATABASE IF NOT EXISTS {database}"

for database in databases:
    query = base_query.format(database=database)
    logger.info(f"SQL: {query}")
    spark.sql(query.format(database=database))

In [0]:
def get_schema(schema_path):
    with open(schema_path) as f:
        return StructType.fromJson(json.load(f))

In [0]:
target_tables = ["expedia_raw", "hotel_weather_raw"]
partition_cols_table_map = {
    "expedia_raw": [], 
    "hotel_weather_raw": ["year", "month", "day"]
}
database = "bronze"

for target_table in target_tables:
    if spark.catalog.tableExists(f"{database}.{target_table}"):
        logger.info(f"TABLE ALREADY EXISTS = {database}.{target_table}")
        continue
    query = f"CREATE TABLE {database}.{target_table} PARTITIONED BY ({', '.join(partition_cols_table_map.get(target_table))})"
    schema = get_schema(f"schemas/{target_table}.json")
    logger.info(f"SQL: {query}")
    empty_df = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema)
    empty_df.write \
        .format("delta") \
        .partitionBy(*partition_cols_table_map.get(target_table)) \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{database}.{target_table}") 